# Hirata et al. (2020) Al–Sc–N CALPHAD — Python reimplementation

K. Hirata, K. Shobu, H. Yamada, M. Uehara, S. A. Anggraini, M. Akiyama,
*"Thermodynamic assessment of the Al–Sc–N ternary system and phase-separated region of the
strained wurtzite phase,"* **J. Eur. Ceram. Soc. 40 (2020) 5410–5422**,
doi:[10.1016/j.jeurceramsoc.2020.06.047](https://doi.org/10.1016/j.jeurceramsoc.2020.06.047)

Companion to `Hirata2020_AlScN_CALPHAD.nb` (Mathematica) and
`hirata2020_methodology_review.md` (the critique).

**Runs on the Python standard library alone.** `matplotlib` is optional and only used for plots.

---

## What this notebook establishes

| | |
|---|---|
| **Reproduced exactly** | Table 1 SGTE lattice stabilities; Table 2 assessed parameters; Redlich–Kister mixing; the miscibility gap; the strain-energy *relaxation factor* |
| **Reconstructed** | Every algebraic **sign** in Tables 1–2 — the PDF text layer drops minus glyphs. Reconstruction is *proven*, not asserted: H − H_SER = 0 at 298.15 K and S(298.15) match SGTE reference values that were never inputs |
| **Not reproducible** | The Debye–Grüneisen chain and the strain energy — the paper never tabulates the inputs (see §8) |

## Two headline findings

1. **The often-quoted T_c = L⁰/2R ≈ 5892 K is not this database's critical temperature.** Solving
   d²G/dx² = d³G/dx³ = 0 for the full three-term Redlich–Kister gives **T_c = 4944 K at x_c = 0.317**.
   The regular-solution shortcut overstates T_c by 19 % *and* misplaces the apex from 0.317 to 0.5.
2. **Table 2 and Table 3 disagree on the AlN wurtzite formation enthalpy by 8.50 kJ/mol-atom (5.4 %).**
   The same expression's *T*-dependent coefficients reproduce Table 3's own entropy to 0.2 %, so the
   discrepancy sits entirely in the leading constant.

## 2. Conventions: units and the per-formula-unit vs per-atom trap

CALPHAD mixes *per mole of formula unit* with *per mole of atoms*, and this is the single largest
source of factor-of-two errors when reading this paper. Conventions used throughout:

- Table 2 is **"J/mol of model"**. For wurtzite the model is `(Al,Sc)(N)` = **1 formula unit = 2 atoms**.
- Table 3 labels its column **"kJ/mol of atom"** — correct.
- Table 4 heads its formation-enthalpy block **"kJ/mol"** — **wrong by a factor of two**; the value
  −185.5 only matches the Table 2 Gibbs energy on a *per-atom* reading.

Every quantity below carries its basis in the variable name.

In [ ]:
import math

GAS_CONSTANT = 8.314462618            # J / (mol K)
ROOM_TEMPERATURE = 298.15             # K
JOULE_PER_ELECTRONVOLT = 96485.33212  # J/mol per eV/atom

ATOMS_PER_FORMULA_UNIT_WURTZITE = 2   # (Al,Sc)(N)
ATOMS_PER_FORMULA_UNIT_ROCKSALT = 2   # (Al,Sc)(N,Va) on the N-full end


def joules_per_formula_unit_to_millielectronvolt_per_atom(value_joule_per_formula_unit,
                                                          atoms_per_formula_unit=2):
    """J/mol-formula-unit -> meV/atom."""
    return (value_joule_per_formula_unit / atoms_per_formula_unit
            / JOULE_PER_ELECTRONVOLT * 1000.0)

## 3. Cross-check harness

Every quantity that is **both** stated in the paper **and** derivable from other published values is
recomputed from the raw parameters and asserted against the stated value. Three checks are
**expected to fail** — they are the internal inconsistencies of §9, and they are marked
`expect_failure=True` so a clean run still reports `ALL AS EXPECTED`.

In [ ]:
CHECK_RESULTS = []


def check_value(computed, expected, tolerance_relative, label,
                expect_failure=False, note=""):
    """Assert computed ~= expected to a relative tolerance; record PASS/FAIL."""
    if expected == 0:
        relative_error = abs(computed)
    else:
        relative_error = abs(computed - expected) / abs(expected)
    passed = relative_error <= tolerance_relative
    as_expected = (passed != expect_failure)
    CHECK_RESULTS.append({"label": label, "computed": computed, "expected": expected,
                          "relative_error": relative_error, "passed": passed,
                          "expect_failure": expect_failure, "as_expected": as_expected,
                          "note": note})
    tag = "PASS" if passed else "FAIL"
    if expect_failure:
        tag += " (expected FAIL)" if not passed else " (UNEXPECTED PASS)"
    print(f"[{tag:22s}] {label}\n"
          f"{'':24s} computed={computed:.6g}  expected={expected:.6g}  rel.err={relative_error:.3g}"
          + (f"\n{'':24s} {note}" if note else ""))
    return passed


def check_zero(computed, absolute_tolerance, label, note=""):
    passed = abs(computed) <= absolute_tolerance
    CHECK_RESULTS.append({"label": label, "computed": computed, "expected": 0.0,
                          "relative_error": abs(computed), "passed": passed,
                          "expect_failure": False, "as_expected": passed, "note": note})
    print(f"[{'PASS' if passed else 'FAIL':22s}] {label}\n"
          f"{'':24s} computed={computed:.6g}  |tol|={absolute_tolerance:.3g}"
          + (f"\n{'':24s} {note}" if note else ""))
    return passed


def summarise_checks():
    total = len(CHECK_RESULTS)
    as_expected = sum(1 for r in CHECK_RESULTS if r["as_expected"])
    print(f"\n{'='*78}\n{as_expected}/{total} checks behaved as expected.")
    unexpected = [r for r in CHECK_RESULTS if not r["as_expected"]]
    if unexpected:
        print("UNEXPECTED:")
        for r in unexpected:
            print(f"  - {r['label']}  (rel.err {r['relative_error']:.3g})")
    else:
        print("ALL AS EXPECTED (including the two deliberate failures of section 9).")

## 4. Table 1 — SGTE lattice stabilities, and proving the sign reconstruction

Standard SGTE polynomial, J per mole of atoms:

$$G(T) = a + bT + cT\ln T + dT^{2} + eT^{-1} + fT^{3}$$

from which

$$S = -\frac{dG}{dT}, \qquad H = G - T\frac{dG}{dT} = a - cT - dT^{2} + \frac{2e}{T} - 2fT^{3}$$

**Why this section matters.** The PDF text layer drops every minus glyph, so all signs had to be
reconstructed. The reconstruction is *provable*: for the SGTE reference states,
`H(298.15) − H_SER` must be **exactly zero** and `S(298.15)` must match the tabulated standard
entropies — neither of which was used as an input.

In [ ]:
def sgte_gibbs_energy(coefficients, temperature):
    """G(T) = a + b T + c T lnT + d T^2 + e/T + f T^3   [J per mol of atoms]."""
    a, b, c, d, e, f = coefficients
    return (a + b * temperature + c * temperature * math.log(temperature)
            + d * temperature ** 2 + e / temperature + f * temperature ** 3)


def sgte_entropy(coefficients, temperature):
    a, b, c, d, e, f = coefficients
    return -(b + c * (math.log(temperature) + 1.0) + 2.0 * d * temperature
             - e / temperature ** 2 + 3.0 * f * temperature ** 2)


def sgte_enthalpy(coefficients, temperature):
    a, b, c, d, e, f = coefficients
    return (a - c * temperature - d * temperature ** 2
            + 2.0 * e / temperature - 2.0 * f * temperature ** 3)


# ---- SGTE reference states (Dinsdale 1991), 298.15 K < T < 700 / 800 K branches ----
# (a, b, c, d, e, f)
GIBBS_ALUMINIUM_FCC_LOW = (-7976.15, 137.093038, -24.3671976,
                           -1.884662e-3, 74092.0, -0.877664e-6)
GIBBS_SCANDIUM_HCP_LOW = (-8689.547, 153.48097, -28.1882,
                          +3.21892e-3, 72177.0, -1.64531e-6)
GIBBS_HALF_NITROGEN_GAS_LOW = (-3750.675, -9.45425, -12.7819,
                               -1.76686e-3, -32374.0, 2.681e-9)

print("SGTE reference states at 298.15 K "
      "(H - H_SER must vanish; S must match the standard entropy)\n")
for name, coefficients, standard_entropy in [
        ("Al fcc  ", GIBBS_ALUMINIUM_FCC_LOW, 28.30),
        ("Sc hcp  ", GIBBS_SCANDIUM_HCP_LOW, 34.64),
        ("0.5 N2  ", GIBBS_HALF_NITROGEN_GAS_LOW, 191.61 / 2)]:
    enthalpy = sgte_enthalpy(coefficients, ROOM_TEMPERATURE)
    entropy = sgte_entropy(coefficients, ROOM_TEMPERATURE)
    print(f"  {name}  H-H_SER = {enthalpy:12.4f} J/mol      "
          f"S = {entropy:9.4f}  (SGTE {standard_entropy:.2f})")

## 5. Table 2 — the assessed parameters

The values that matter for everything downstream. Note the wurtzite ternary gets **three**
Redlich–Kister terms while the rocksalt (FCC) ternary gets **one** — an asymmetry that biases the
wurtzite/rocksalt crossover, since the crossover is a *difference* of two Gibbs curves.

Note also that **every ternary L is a bare constant with no temperature coefficient**, so the excess
entropy of mixing (i.e. the mixing-induced excess vibrational entropy) is identically zero.

In [ ]:
# --- Table 2, Al-Sc-N ternary. J per mol of model; model = (Al,Sc)(N) = 1 f.u. = 2 atoms
REDLICH_KISTER_WURTZITE = (97970.4838186,     # L^0
                           434.322298892,     # L^1
                           17960.9058372)     # L^2
REDLICH_KISTER_ROCKSALT = (90970.6587083,)    # L^0 only -- the paper fits no more

# --- Table 2, wurtzite endmembers. J per mol of formula unit.
# Signs reconstructed; proven below against Table 3's entropy.
GIBBS_ALUMINIUM_NITRIDE_WURTZITE = (-315445.36, 291.78238, -45.177271,
                                    -0.00285744685, 683276.5, 1.69215267e-17)
GIBBS_SCANDIUM_NITRIDE_WURTZITE = (-347780.61, 285.610224, -48.138732,
                                   -0.0011557218, 295515.2, 0.0)

# --- Sc-N rocksalt (FCC) vacancy interaction -- the ONLY vacancy modelling in the paper
REDLICH_KISTER_SCANDIUM_NITRIDE_FCC_VACANCY = (20606.0368145, 22501.84214)

# --- Tables 3 and 4, as printed
ALUMINIUM_NITRIDE_ENTHALPY_TABLE3_KJ_PER_MOL_ATOM = -157.07
ALUMINIUM_NITRIDE_ENTROPY_TABLE3 = 10.07          # J/(mol-atom K)
SCANDIUM_NITRIDE_ENTHALPY_TABLE4 = -185.5         # kJ/mol -- per ATOM despite the label
SCANDIUM_NITRIDE_LATTICE_CONSTANT_NM = 0.452
ALUMINIUM_SCANDIUM_NITRIDE_LATTICE_CONSTANT_NM = 0.442

print("wurtzite  L0, L1, L2 =", REDLICH_KISTER_WURTZITE, "J/mol-f.u.")
print("rocksalt  L0         =", REDLICH_KISTER_ROCKSALT, "J/mol-f.u.")

## 6. Redlich–Kister mixing, and the real critical point

For $\mathrm{Al}_{1-x}\mathrm{Sc}_x\mathrm{N}$ with $y_{\mathrm{Al}} = 1-x$, $y_{\mathrm{Sc}} = x$:

$$G^{\mathrm{ex}}(x) = x(1-x)\sum_{n} L^{(n)}\,(1-2x)^{n}$$
$$G^{\mathrm{mix}}(x,T) = G^{\mathrm{ex}}(x) + RT\big[x\ln x + (1-x)\ln(1-x)\big]$$

The critical point satisfies $\partial^2 G/\partial x^2 = \partial^3 G/\partial x^3 = 0$.

**This is where the widely-quoted number is wrong.** $L^{(2)}$ is a fifth of $L^{(0)}$ and flattens
the top of the dome enough to move where the spinodal first opens — from $x=0.5$ to $x\approx0.317$,
and from 5892 K down to 4944 K. The regular-solution shortcut $L^{(0)}/2R$ is only exact when
$L^{(1)}=L^{(2)}=0$, which is true for the **rocksalt** phase here but not the wurtzite one.

The apex lands at $x \approx 0.32$ — inside the composition window of interest for ferroelectric
AlScN — and it lands there because of the $L^{(2)}$ term the paper fitted straight through the
wurtzite → h-BN structural transition. Treat the feature as a probable artifact of that fit.

In [ ]:
def excess_gibbs_energy(scandium_fraction, redlich_kister_coefficients):
    """Redlich-Kister excess Gibbs energy, J per mol of formula unit."""
    x = scandium_fraction
    asymmetry = 1.0 - 2.0 * x
    series = sum(coefficient * asymmetry ** order
                 for order, coefficient in enumerate(redlich_kister_coefficients))
    return x * (1.0 - x) * series


def ideal_mixing_entropy(scandium_fraction):
    """Configurational entropy on the CATION sublattice only, J/(mol-f.u. K).

    Only the cation sublattice mixes, so per ATOM this is half the usual R ln2 --
    0.347 kB/atom, not 0.693. That halving is why the excess VIBRATIONAL entropy of
    mixing (0.1-0.2 kB/atom in substitutional alloys) is such a large fraction of the
    total here, and why omitting it matters.
    """
    x = scandium_fraction
    if x <= 0.0 or x >= 1.0:
        return 0.0
    return -GAS_CONSTANT * (x * math.log(x) + (1.0 - x) * math.log(1.0 - x))


def gibbs_energy_of_mixing(scandium_fraction, temperature, redlich_kister_coefficients):
    return (excess_gibbs_energy(scandium_fraction, redlich_kister_coefficients)
            - temperature * ideal_mixing_entropy(scandium_fraction))


def second_derivative_of_mixing(scandium_fraction, temperature,
                                redlich_kister_coefficients, step=1e-5):
    x, h = scandium_fraction, step
    return (gibbs_energy_of_mixing(x + h, temperature, redlich_kister_coefficients)
            - 2.0 * gibbs_energy_of_mixing(x, temperature, redlich_kister_coefficients)
            + gibbs_energy_of_mixing(x - h, temperature, redlich_kister_coefficients)) / h ** 2


def spinodal_interval(temperature, redlich_kister_coefficients, sample_count=20001):
    """Composition range over which d2G/dx2 < 0, or None if the alloy is stable."""
    unstable = [1e-4 + (1.0 - 2e-4) * i / (sample_count - 1)
                for i in range(sample_count)
                if second_derivative_of_mixing(1e-4 + (1.0 - 2e-4) * i / (sample_count - 1),
                                               temperature, redlich_kister_coefficients) < 0.0]
    return (min(unstable), max(unstable)) if unstable else None


def critical_temperature(redlich_kister_coefficients,
                         lower_bound=1000.0, upper_bound=9000.0, iterations=60):
    """Highest T with any spinodal region, plus the composition where it closes."""
    low, high = lower_bound, upper_bound
    for _ in range(iterations):
        middle = 0.5 * (low + high)
        if spinodal_interval(middle, redlich_kister_coefficients, 4001):
            low = middle
        else:
            high = middle
    interval = spinodal_interval(low, redlich_kister_coefficients, 200001)
    critical_composition = 0.5 * (interval[0] + interval[1]) if interval else float("nan")
    return low, critical_composition


wurtzite_critical_temperature, wurtzite_critical_composition = \
    critical_temperature(REDLICH_KISTER_WURTZITE)
regular_solution_estimate = REDLICH_KISTER_WURTZITE[0] / (2.0 * GAS_CONSTANT)
rocksalt_critical_temperature = REDLICH_KISTER_ROCKSALT[0] / (2.0 * GAS_CONSTANT)

print(f"wurtzite, FULL three-term RK : T_c = {wurtzite_critical_temperature:7.1f} K "
      f"at x_c = {wurtzite_critical_composition:.4f}")
print(f"wurtzite, regular-solution   : T_c = {regular_solution_estimate:7.1f} K at x   = 0.5"
      f"   <-- overstates by "
      f"{(regular_solution_estimate/wurtzite_critical_temperature-1)*100:.0f}%")
print(f"rocksalt, L0 only (exact)    : T_c = {rocksalt_critical_temperature:7.1f} K at x   = 0.5")
print("\nSpinodal interval versus temperature:")
for temperature in (3000, 4000, 4500, 4900, 4940, 4944, 5000, 5892):
    interval = spinodal_interval(temperature, REDLICH_KISTER_WURTZITE)
    print(f"  T = {temperature:5d} K  ->  "
          + (f"{interval[0]:.4f} .. {interval[1]:.4f}" if interval else "single phase"))

## 7. Equations (20)–(22) — epitaxial strain energy

$$G_{\mathrm{strain}} = 2V\mu\frac{1+\nu}{1-\nu}\,\varepsilon^{2} \qquad (h < h_c)$$
$$G_{\mathrm{strain}} = 2V\mu\frac{1+\nu}{1-\nu}\,\varepsilon^{2}\cdot\frac{h_c}{h}\left(1+\ln\frac{h}{h_c}\right) \qquad (h > h_c)$$

The **prefactor cannot be evaluated** — $\mu(x)$, $\nu(x)$, $V(x)$, $\varepsilon(x)$ and $h_c(x)$ are
none of them tabulated (§8). The **relaxation factor** is exact and needs no paper-specific inputs,
and it is what carries the paper's central claim.

With $h_c \approx 2$ nm (Zhang *et al.*, JAP 114, 243516, for $\mathrm{Sc}_{0.375}\mathrm{Al}_{0.625}\mathrm{N}$
on AlN), all three thicknesses the paper models are deep in the dislocation-relaxed regime — and real
films are 1–2 µm.

### The analytic decomposition that matters for the strain/Ω degeneracy

With a linear Vegard misfit referenced to the substrate, $\varepsilon(x) = cx$, and $x^2 = x - x(1-x)$:

$$G_{\mathrm{strain}} = K(h)c^{2}x^{2} = \underbrace{K(h)c^{2}x}_{\text{endmember tilt}} - \underbrace{K(h)c^{2}\,x(1-x)}_{\Delta L^{(0)} = -K(h)c^{2}}$$

So epitaxial strain is *formally identical* to a thickness-dependent interaction parameter. Ω-calibration
and a strain term are therefore **degenerate against a single measured crossover composition**.

In [ ]:
def strain_relaxation_factor(film_thickness_nm, critical_thickness_nm):
    """Fraction of the fully-coherent strain energy retained at a given thickness."""
    if film_thickness_nm <= critical_thickness_nm:
        return 1.0
    ratio = critical_thickness_nm / film_thickness_nm
    return ratio * (1.0 + math.log(film_thickness_nm / critical_thickness_nm))


CRITICAL_THICKNESS_NM = 2.0   # Zhang et al., JAP 114, 243516 -- NOT from Hirata
print(f"Relaxation factor f(h) with h_c = {CRITICAL_THICKNESS_NM} nm\n")
reference_factor = strain_relaxation_factor(10.0, CRITICAL_THICKNESS_NM)
for thickness_nm in (2.0, 10.0, 50.0, 100.0, 1000.0, 2000.0):
    factor = strain_relaxation_factor(thickness_nm, CRITICAL_THICKNESS_NM)
    label = "  <-- the paper models these" if thickness_nm in (10.0, 50.0, 100.0) else (
            "  <-- REAL FILMS" if thickness_nm >= 1000.0 else "")
    print(f"  h = {thickness_nm:7.0f} nm   f = {factor:.5f}   "
          f"({reference_factor/factor:5.1f}x weaker than at 10 nm){label}")

## 8. GAPS — what the paper never parametrizes

Verified against the source. These are why Figs. 12/13/14 are **structurally** reproducible but not
**numerically** reproducible.

| Missing | Consequence |
|---|---|
| **Morse parameters** $A$, $D$, $\lambda$, $r_0$ — never tabulated for any of the five compounds, nor the $E(V)$ points they were fitted to | **Eqs. (2)–(13), the entire Debye–Grüneisen chain, cannot be evaluated with the paper's own inputs** |
| $\mu(x)$ and $\nu(x)$ — only curves in Fig. 10(b),(c) | **Eqs. (20)–(21) cannot be evaluated quantitatively** |
| $h_c(x)$ — imported from Zhang [38], neither tabulated nor plotted | strain model depends on an external number |
| $a(x)$, $c(x)$ — only Fig. 11 | the misfit $\varepsilon(x)$ is not computable |
| molar volume $V(x)$ | never given at all |
| **wurtzite vacancy endmembers** $^{\circ}G_{\mathrm{Al:Va}}$, $^{\circ}G_{\mathrm{Sc:Va}}$ | the "(Al,Sc)(N,Va)" label in Table 2 is an **empty label**; wurtzite AlN and (Al,Sc)N are strict line compounds with zero homogeneity width |
| the 128-atom **SQS structures** | Fig. 8 cannot be recomputed, only the fitted curves |
| the **per-thickness refitted L** of Fig. 13 | not tabulated |
| any excess entropy of mixing (the mixing-induced excess vibrational entropy) | every ternary L is a bare constant, no T coefficient |
| any **μ_N or P(N₂) axis** | ½E(N₂) is a fixed reference |
| an **h-BN-like polymorph** | absent, despite the paper reporting the c/a collapse and calling x = 32.5 % an outlier on that basis |

The placeholders below are **stand-ins, not from this paper**, provided only so downstream cells
evaluate.

In [ ]:
# ################  PLACEHOLDERS -- NOT FROM HIRATA ET AL.  ################
# Provided only so that the strain cells evaluate end to end. Replace before use.
PLACEHOLDER_WARNING = ("STAND-IN VALUE, not from Hirata et al. (2020) -- "
                       "digitize Figs. 10/11 or recompute before relying on this.")

def placeholder_shear_modulus_pascal(scandium_fraction):
    """~130 GPa for AlN softening toward ~90 GPa by x = 0.3. STAND-IN."""
    return (130.0 - 130.0 * scandium_fraction) * 1e9


def placeholder_poisson_ratio(scandium_fraction):
    """0.22 for AlN rising toward 0.40 by x = 0.375 (Zhang). STAND-IN."""
    return 0.22 + (0.40 - 0.22) * min(scandium_fraction / 0.375, 1.0)


def placeholder_misfit_strain(scandium_fraction):
    """Vegard between a(AlN) = 3.112 A and a(wz-ScN) ~ 3.65 A, on an AlN reference. STAND-IN."""
    return (3.112 + scandium_fraction * (3.65 - 3.112) - 3.112) / 3.112


PLACEHOLDER_MOLAR_VOLUME_CUBIC_METRE_PER_MOL = 1.259e-5   # AlN wurtzite. STAND-IN.

print("!! " + PLACEHOLDER_WARNING)

In [ ]:
def strain_energy_joule_per_formula_unit(scandium_fraction, film_thickness_nm,
                                        critical_thickness_nm=CRITICAL_THICKNESS_NM):
    """Eqs. (20)-(21) evaluated with PLACEHOLDER elastic data. Trend only."""
    shear_modulus = placeholder_shear_modulus_pascal(scandium_fraction)
    poisson_ratio = placeholder_poisson_ratio(scandium_fraction)
    misfit = placeholder_misfit_strain(scandium_fraction)
    coherent_energy = (2.0 * PLACEHOLDER_MOLAR_VOLUME_CUBIC_METRE_PER_MOL * shear_modulus
                       * (1.0 + poisson_ratio) / (1.0 - poisson_ratio) * misfit ** 2)
    return coherent_energy * strain_relaxation_factor(film_thickness_nm, critical_thickness_nm)


print("Strain energy vs mixing enthalpy at x = 0.30   (PLACEHOLDER elastic data)\n")
mixing_enthalpy = excess_gibbs_energy(0.30, REDLICH_KISTER_WURTZITE)
print(f"  mixing enthalpy (Table 2, real)     = {mixing_enthalpy/1000:7.2f} kJ/mol-f.u.")
for thickness_nm in (10.0, 100.0, 1000.0):
    strain_energy = strain_energy_joule_per_formula_unit(0.30, thickness_nm)
    print(f"  strain energy at h = {thickness_nm:6.0f} nm      = {strain_energy/1000:7.2f} "
          f"kJ/mol-f.u.   ({strain_energy/mixing_enthalpy*100:5.1f} % of mixing enthalpy)")
print("\n  For scale: the OMITTED excess vibrational free energy of mixing at ~1000 K is")
print("  roughly 2.5 kJ/mol-f.u. at x = 0.5 -- larger than the strain term at every real thickness.")

## 9. Run every cross-check

Two failures are **deliberate** — they are the paper's internal inconsistencies.

In [ ]:
CHECK_RESULTS.clear()

# ---- SGTE reference states: H - H_SER must vanish, S must match ----
check_zero(sgte_enthalpy(GIBBS_ALUMINIUM_FCC_LOW, ROOM_TEMPERATURE), 0.05,
           "Al fcc: H(298.15) - H_SER = 0  [proves the sign reconstruction]")
check_zero(sgte_enthalpy(GIBBS_SCANDIUM_HCP_LOW, ROOM_TEMPERATURE), 0.05,
           "Sc hcp: H(298.15) - H_SER = 0  [proves the sign reconstruction]")
check_value(sgte_entropy(GIBBS_ALUMINIUM_FCC_LOW, ROOM_TEMPERATURE), 28.30, 0.002,
            "Al fcc: S(298.15) vs SGTE 28.30 J/(mol K)")
check_value(sgte_entropy(GIBBS_SCANDIUM_HCP_LOW, ROOM_TEMPERATURE), 34.64, 0.002,
            "Sc hcp: S(298.15) vs SGTE 34.64 J/(mol K)")
check_zero(sgte_enthalpy(GIBBS_HALF_NITROGEN_GAS_LOW, ROOM_TEMPERATURE), 0.05,
           "0.5 N2 gas: H(298.15) - H_SER = 0  [proves the sign reconstruction]")
check_value(sgte_entropy(GIBBS_HALF_NITROGEN_GAS_LOW, ROOM_TEMPERATURE), 191.61 / 2, 0.002,
            "0.5 N2 gas: S(298.15) vs half the SGTE 191.61 J/(mol K)")

# ---- Table 2 wurtzite AlN against Table 3 ----
aluminium_nitride_entropy = sgte_entropy(GIBBS_ALUMINIUM_NITRIDE_WURTZITE,
                                         ROOM_TEMPERATURE) / ATOMS_PER_FORMULA_UNIT_WURTZITE
check_value(aluminium_nitride_entropy, ALUMINIUM_NITRIDE_ENTROPY_TABLE3, 0.005,
            "AlN wurtzite S(298.15) from Table 2 vs Table 3's 10.07 J/(mol-atom K)",
            note="the T-dependent coefficients are correct")

aluminium_nitride_enthalpy = (sgte_enthalpy(GIBBS_ALUMINIUM_NITRIDE_WURTZITE, ROOM_TEMPERATURE)
                              / ATOMS_PER_FORMULA_UNIT_WURTZITE / 1000.0)
check_value(aluminium_nitride_enthalpy, ALUMINIUM_NITRIDE_ENTHALPY_TABLE3_KJ_PER_MOL_ATOM, 0.01,
            "AlN wurtzite dH(298.15) from Table 2 vs Table 3's -157.07 kJ/mol-atom",
            expect_failure=True,
            note="DELIBERATE: Table 2 and Table 3 disagree by 8.50 kJ/mol-atom (5.4%)")

# ---- Redlich-Kister sanity ----
check_zero(excess_gibbs_energy(0.0, REDLICH_KISTER_WURTZITE), 1e-9,
           "Redlich-Kister vanishes at x = 0")
check_zero(excess_gibbs_energy(1.0, REDLICH_KISTER_WURTZITE), 1e-9,
           "Redlich-Kister vanishes at x = 1")

# ---- the headline mixing / critical-point numbers ----
check_value(excess_gibbs_energy(0.5, REDLICH_KISTER_WURTZITE),
            REDLICH_KISTER_WURTZITE[0] / 4.0, 1e-12,
            "dH_mix(wurtzite, x=0.5) = L0/4 = 24492.6 J/mol-f.u.")
check_value(joules_per_formula_unit_to_millielectronvolt_per_atom(
                excess_gibbs_energy(0.5, REDLICH_KISTER_WURTZITE)), 126.92, 0.001,
            "dH_mix(wurtzite, x=0.5) in meV/atom")
check_value(ideal_mixing_entropy(0.5), GAS_CONSTANT * math.log(2.0), 1e-12,
            "ideal mixing entropy at x=0.5 = R ln2 (per mol CATION sites)")
check_value(ideal_mixing_entropy(0.5) / ATOMS_PER_FORMULA_UNIT_WURTZITE
            / (GAS_CONSTANT), 0.3466, 0.001,
            "... = 0.347 kB/atom, HALF the usual 0.693 because only cations mix")
check_value(wurtzite_critical_temperature, 4943.6, 0.005,
            "wurtzite T_c from the FULL three-term RK",
            note="NOT 5892 K -- that is the regular-solution shortcut")
check_value(wurtzite_critical_composition, 0.3172, 0.01,
            "wurtzite x_c -- the apex is NOT at x = 0.5")
check_value(rocksalt_critical_temperature, 5470.6, 0.001,
            "rocksalt T_c = L0/2R (exact here: only L0 was fitted)")

# ---- strain relaxation factor ----
for thickness_nm, expected in ((10.0, 0.52189), (50.0, 0.16876),
                               (100.0, 0.09824), (1000.0, 0.014431)):
    check_value(strain_relaxation_factor(thickness_nm, 2.0), expected, 1e-3,
                f"strain relaxation factor at h = {thickness_nm:.0f} nm")

# ---- the strain <-> interaction-parameter identity behind dL0 = -K ----
_stiffness = 12345.0
for _x in (0.15, 0.4, 0.77):
    check_zero(_stiffness * _x ** 2
               - (_stiffness * _x - _stiffness * _x * (1.0 - _x)), 1e-9,
               f"identity K x^2 = K x - K x(1-x) at x = {_x}  [the basis of dL0 = -K]")

# ---- Table 4 unit label ----
check_value(SCANDIUM_NITRIDE_ENTHALPY_TABLE4 * 2.0, -185.5, 0.01,
            "ScN Table 4 read as per-FORMULA-UNIT",
            expect_failure=True,
            note="DELIBERATE: only the per-ATOM reading is consistent; the label is wrong by 2x")

summarise_checks()

## 10. Plots (optional — needs matplotlib)

Skipped cleanly if matplotlib is unavailable.

In [ ]:
try:
    import matplotlib.pyplot as plt
    HAVE_MATPLOTLIB = True
except ImportError:
    HAVE_MATPLOTLIB = False
    print("matplotlib not installed -- skipping plots. The numerical results above are unaffected.")

if HAVE_MATPLOTLIB:
    compositions = [i / 400.0 for i in range(401)]
    figure, (axis_left, axis_right) = plt.subplots(1, 2, figsize=(12, 4.5))

    axis_left.plot(compositions,
                   [excess_gibbs_energy(x, REDLICH_KISTER_WURTZITE) / 1000.0
                    for x in compositions], label="wurtzite (3-term RK)")
    axis_left.plot(compositions,
                   [excess_gibbs_energy(x, REDLICH_KISTER_ROCKSALT) / 1000.0
                    for x in compositions], "--", label="rocksalt (L0 only)")
    axis_left.set_xlabel("Sc fraction x"); axis_left.set_ylabel("dH_mix  [kJ/mol-f.u.]")
    axis_left.set_title("Mixing enthalpy (the paper's Fig. 8)")
    axis_left.legend(); axis_left.grid(alpha=0.3)

    temperatures = list(range(2000, 5200, 50))
    lower_branch, upper_branch, plotted = [], [], []
    for temperature in temperatures:
        interval = spinodal_interval(temperature, REDLICH_KISTER_WURTZITE, 4001)
        if interval:
            plotted.append(temperature)
            lower_branch.append(interval[0]); upper_branch.append(interval[1])
    axis_right.plot(lower_branch, plotted, "b-")
    axis_right.plot(upper_branch, plotted, "b-")
    axis_right.axhline(wurtzite_critical_temperature, color="r", ls=":",
                       label=f"T_c = {wurtzite_critical_temperature:.0f} K "
                             f"at x = {wurtzite_critical_composition:.3f}")
    axis_right.axhline(regular_solution_estimate, color="grey", ls="--",
                       label=f"regular-solution L0/2R = {regular_solution_estimate:.0f} K")
    axis_right.set_xlabel("Sc fraction x"); axis_right.set_ylabel("Temperature [K]")
    axis_right.set_title("Wurtzite spinodal"); axis_right.legend(fontsize=8)
    axis_right.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 11. Findings — errors and inconsistencies in the published parameters

1. **Table 2 and Table 3 disagree on AlN wurtzite ΔH by 8.50 kJ/mol-atom (5.4 %).**
   Table 2's expression gives −148.57 kJ/mol-atom; Table 3 states −157.07. The *same* expression's
   temperature-dependent coefficients reproduce Table 3's own entropy to 0.2 %, so the discrepancy
   sits **entirely in the leading constant**: −332 447 J/mol would reproduce Table 3 exactly,
   against the −315 445.36 printed. The ScN entry treated identically matches Table 4 to 0.02 %, so
   this is not a systematic misreading — it is a typo in one of the two tables.

2. **Table 4's unit label is wrong by a factor of two** — it heads the block "kJ/mol", but −185.5
   only matches the Table 2 Gibbs energy on a per-**atom** reading. Table 3 labels its column correctly.

3. **Eq. (11) is dimensionally inconsistent with Eq. (13).** As printed, α = (1/r₀)dr₀/dT is the
   *linear* expansion coefficient, but C_p = C_v + α²B₀V₀T requires the *volumetric* one. Taken
   literally the pair understates the C_p correction by a factor of 9.

4. **T_c = L⁰/2R ≈ 5892 K is not this database's critical temperature** — the full three-term
   Redlich–Kister gives **4944 K at x_c = 0.317**. The shortcut overstates by 19 % and misplaces the
   apex. Rocksalt's 5471 K *is* correct, because only L⁰ was fitted there.

5. **The crossover composition the paper never states**: its own database puts the wurtzite/rocksalt
   crossover at x ≈ 0.60 at 298 K — between Zhang's 0.56 (no vdW) and Talley's 0.64 (optPBE-vdW).
   §7 calls its enthalpies "relatively consistent" with experiment without ever quoting this number.

6. **Smaller:** AlSc₃N's formation enthalpy is printed as "104.7 kJ/mol per atom" with the minus sign
   dropped; Eq. (4)'s text calls k(ν) "the derived Poisson's ratio" when it is a *function* of it; and
   Table 2's AlN wurtzite carries a T³ coefficient of 1.69×10⁻¹⁷ contributing <4 µJ/mol at 6000 K —
   an optimiser artifact.

### Model-level criticisms (not transcription issues)

See `hirata2020_methodology_review.md` and `calphad_strategy_from_hirata_critique.md`. In brief: no
excess vibrational entropy of mixing (worth ≈ −37 % on T_c, larger than the strain effect the paper is
built around); no short-range order (a further ≈ −20–30 %); one "wurtzite" phase fitted straight
through the h-BN structural transition; and no nitrogen off-stoichiometry in wurtzite at all.